# Yeom MEG 3D-reaching — Colab GPU runner

Trains the Brain2Qwerty-style **`convtransformer`** decoder on the Yeom 2023 MEG
reaching dataset using a Colab **NVIDIA GPU**.

**Flow:** mount Drive → download dataset → extract your subject to Drive → load → train.

**Before running:**
1. *Runtime → Change runtime type → Hardware accelerator: **GPU*** (a T4 is plenty).
2. The code repo is **public**, so the *Get the code* cell clones it with no token.

The per-subject `.mat` you extract is written to Drive, so re-running in a fresh
session skips the download. The Yeom `.mat` are MATLAB **v7.3** files — `mat73`
(installed below) reads them.

In [ ]:
import torch
!nvidia-smi -L
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> GPU'

In [ ]:
# Colab already has torch(+CUDA), numpy, scipy, scikit-learn. Add the rest:
# - mat73: reads the v7.3 (HDF5) Yeom .mat files
# - mne:   only needed for the optional CSP baseline
# - remotezip: only for the optional small-but-slow download cell
!pip -q install mat73 mne remotezip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the code (public repo — no token)

In [ ]:
import sys, os, subprocess

CODE_SOURCE = 'git'     # 'git' (public repo) or 'drive'
GIT_URL     = 'https://github.com/atticus-429/meg-movement-decoder.git'
DRIVE_CODE_DIR = '/content/drive/MyDrive/hcp_motor_decoder'   # used only if CODE_SOURCE='drive'

if CODE_SOURCE == 'git':
    if not os.path.isdir('/content/code'):
        subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, '/content/code'], check=True)
    PKG = '/content/code'          # repo root IS the package
else:
    PKG = DRIVE_CODE_DIR
assert os.path.isdir(PKG), f'code folder not found: {PKG}'
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import config, decode, evaluate
from loaders.yeom import load_yeom
print('code OK at', PKG)

## 2. Configure dataset + download target

In [ ]:
# dataset variant: 'ica' (artifact-cleaned, recommended) or 'epoched'
DATA_VARIANT  = 'ica'
FIGSHARE_FILE = {'ica': '41898840', 'epoched': '41898714'}[DATA_VARIANT]
ZIP_URL = f'https://ndownloader.figshare.com/files/{FIGSHARE_FILE}'   # one ~9.3 GB zip

# extracted per-subject .mat live here on Drive (persists across sessions)
DRIVE_DATA_DIR = '/content/drive/MyDrive/yeom_meg/data'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# which subject/session to extract: a SUBSTRING of the .mat filename.
# Archive filenames are 'Sub_<n>_ses_<s>_ICA.mat' (ica) / 'Sub_<n>_ses_<s>.mat' (epoched),
# n = 1..9, s = 1..2. The download cell prints the full list.
SUBJECT_TOKEN = 'Sub_1_ses_1'

# keep the big 9.3 GB zip on local disk (fast, ephemeral) -> spares Drive quota.
# set True only if you want to extract many subjects across sessions without re-downloading.
ZIP_TO_DRIVE = False

## 3. Download + extract your subject to Drive

Downloads the full ~9.3 GB archive to **fast local disk** and extracts only the
`.mat` matching `SUBJECT_TOKEN` (~0.5 GB) to Drive. Reliable, resumable (`wget -c`),
and idempotent — if your subject is already on Drive it skips everything.

In [ ]:
import glob, zipfile

def drive_mats():
    fs = glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True)
    return [f for f in fs if (not SUBJECT_TOKEN) or SUBJECT_TOKEN in os.path.basename(f)]

if drive_mats():
    print('already on Drive (skipping download):', [os.path.basename(f) for f in drive_mats()])
else:
    ZIP_PATH = (f'/content/drive/MyDrive/yeom_meg/yeom_{DATA_VARIANT}.zip'
                if ZIP_TO_DRIVE else f'/content/yeom_{DATA_VARIANT}.zip')
    os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
    if not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 9_000_000_000):
        print(f'downloading {DATA_VARIANT} archive (~9.3 GB) -> {ZIP_PATH} ...')
        rc = os.system(f'wget -c -O "{ZIP_PATH}" "{ZIP_URL}"')   # -c resumes if interrupted
        assert rc == 0 and os.path.getsize(ZIP_PATH) > 9_000_000_000, \
            'download incomplete — re-run this cell to resume'
    with zipfile.ZipFile(ZIP_PATH) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        print(f'{len(mats)} .mat in archive, e.g. {mats[:3]}')
        sel = [n for n in mats if SUBJECT_TOKEN in n] if SUBJECT_TOKEN else mats
        assert sel, f'no .mat matched SUBJECT_TOKEN={SUBJECT_TOKEN!r}; choose from the list above'
        for n in sel:
            print('extracting', n, '-> Drive'); z.extract(n, DRIVE_DATA_DIR)
    assert drive_mats(), 'extraction produced no .mat on Drive!'
    print('on Drive now:', [os.path.basename(f) for f in drive_mats()])

### (Optional) small download via remotezip

Pulls **only** your subject (~0.5 GB) straight to Drive without the full archive,
but is **~10-15 min** because figshare's download URLs expire every 10 s (each
chunk re-redirects). Use only if you can't spare ~10 GB of local disk. Off by default.

In [ ]:
RUN_REMOTEZIP = False
if RUN_REMOTEZIP and not glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True):
    from remotezip import RemoteZip
    assert SUBJECT_TOKEN, 'set SUBJECT_TOKEN'
    with RemoteZip(ZIP_URL) as z:
        sel = [n for n in z.namelist() if n.lower().endswith('.mat') and SUBJECT_TOKEN in n]
        assert sel, f'no .mat matched {SUBJECT_TOKEN!r}'
        for n in sel:
            print('downloading (slow)', n, '...'); z.extract(n, DRIVE_DATA_DIR)
    print('done')

## 4. Load the subject from Drive

In [ ]:
import numpy as np
X, y, sfreq, names = load_yeom(DRIVE_DATA_DIR, subject=SUBJECT_TOKEN or None,
                              sensor_type='all', resample=150, crop=(0.0, 1.5))
print('X', X.shape, '| classes', names, '| counts', np.bincount(y).tolist(), '| sfreq', sfreq)

## 5. Train the conv→transformer (GPU)

With a GPU, full 5-fold CV is feasible (`CV='kfold'`); use `'holdout'` for a quick
single-split pass. The decoder auto-uses CUDA.

In [ ]:
from decode import build_decoder
from evaluate import bootstrap_error_ci, format_confusion
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

CV = 'kfold'   # 'kfold' (5-fold, GPU) or 'holdout' (single split, fastest)
dec = build_decoder('convtransformer', sfreq=sfreq)

if CV == 'kfold':
    cvs = StratifiedKFold(5, shuffle=True, random_state=0)
    y_pred, y_eval = cross_val_predict(dec, X, y, cv=cvs, n_jobs=1), y
else:
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0)
    dec.fit(X[tr], y[tr]); y_pred, y_eval = dec.predict(X[te]), y[te]

err, lo, hi = bootstrap_error_ci(y_eval, y_pred)
chance = 100.0 / len(names)
print(f'convtransformer | accuracy {100*(1-err):.1f}%  '
      f'95% CI [{100*(1-hi):.1f}, {100*(1-lo):.1f}]  chance {chance:.1f}%')
cs, cm = format_confusion(y_eval, y_pred, names)
print(cs)

os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = '/content/drive/MyDrive/yeom_meg/result_convtransformer.npz'
np.savez(out, y_true=y_eval, y_pred=y_pred, cm=cm, acc=1 - err, classes=names)
print('saved', out)

## 6. (Optional) CSP baseline for comparison

In [ ]:
dec = build_decoder('csp', sfreq=sfreq)
yp = cross_val_predict(dec, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print('csp baseline accuracy:', round(100 * np.mean(yp == y), 1), '%')

## 7. Multi-subject window-comparison suite (GPU)

For each subject-session, runs the chosen decoder under THREE time windows and
aggregates a per-session table + mean +/- spread to Drive:
- **peri**   - cue-locked [0, 1.5 s]            (the original setting)
- **pre**    - accel-gated [onset-0.5 s, onset]  (pre-movement; artifact-controlled)
- **during** - accel-gated [onset, onset+0.5 s]  (execution; equal-length control)

Extracts all 18 `.mat` to local `/content` (~9.3 GB). **WARNING:** the full
18-session x 3-window conv->transformer sweep is long (~1.5-2.5 h on a T4). Set
`SUBSET=['Sub_1']` first for a quick (~3-run) sanity check, comment out windows, or
use `SUITE_DECODER='csp'`.

In [ ]:
# extract ALL subject/session .mat to local disk (not Drive -> spares quota)
import glob, zipfile
ALL_DIR = '/content/yeom_all'
os.makedirs(ALL_DIR, exist_ok=True)
allmats = glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True)
if len(allmats) >= 18:
    print(f'{len(allmats)} .mat already extracted at {ALL_DIR}')
else:
    ZIP_PATH = f'/content/yeom_{DATA_VARIANT}.zip'
    if not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 9_000_000_000):
        print('downloading full archive (~9.3 GB) ...')
        rc = os.system(f'wget -c -O "{ZIP_PATH}" "{ZIP_URL}"')
        assert rc == 0 and os.path.getsize(ZIP_PATH) > 9_000_000_000, 'download incomplete; re-run'
    with zipfile.ZipFile(ZIP_PATH) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        print(f'extracting {len(mats)} .mat to {ALL_DIR} ...')
        for n in mats:
            z.extract(n, ALL_DIR)
    allmats = glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True)
print(f'{len(allmats)} subject-session files ready')

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SUITE_DECODER = 'convtransformer'   # or 'csp'
SENSOR  = 'all'                     # 'all' / 'mag' / 'grad'
SUBSET  = None                      # e.g. ['Sub_1'] for a quick check; None = all 18 sessions
WINDOWS = [                         # (name, align, crop); comment out rows to skip
    ('peri',   'cue',      (0.0, 1.5)),
    ('pre',    'movement', (-0.5, 0.0)),
    ('during', 'movement', (0.0, 0.5)),
]

files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
if SUBSET:
    files = [f for f in files if any(s in os.path.basename(f) for s in SUBSET)]

def label(path):
    m = re.search(r'Sub_\d+_ses_\d+', os.path.basename(path))
    return m.group(0) if m else os.path.basename(path)

cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for f in files:
    name = label(f); rec = {'subject': name}
    for wname, align, crop in WINDOWS:
        X, y, sfreq, names = load_yeom(f, sensor_type=SENSOR, resample=150, crop=crop, align=align)
        dec = build_decoder(SUITE_DECODER, sfreq=sfreq)
        yp = cross_val_predict(dec, X, y, cv=cv, n_jobs=1)
        acc = 100 * np.mean(yp == y)
        rec[wname] = round(acc, 1); rec[wname + '_n'] = int(len(y))
        print(f'{name} [{wname}]: {acc:.1f}%  (n={len(y)})')
    rows.append(rec)

df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
for wname, _, _ in WINDOWS:
    if wname in df:
        print(f'{wname:7s}: mean {df[wname].mean():.1f}% +/- {df[wname].std():.1f}  '
              f'(chance 25%, n={int(df[wname].notna().sum())} sessions)')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/window_comparison_{SUITE_DECODER}_{SENSOR}.csv'
df.to_csv(out, index=False); print('saved', out)